# Demo: Creating a Vector Index in Amazon OpenSearch Serverless

In this demo you will:
1. Create an OpenSearch Serverless vector search collection
2. Create two vector indexes for different knowledge domains
3. Load text documents, chunk them, and generate embeddings
4. Execute similarity queries against each index

## Step 1: Install Dependencies

In [ ]:
!pip install opensearch-py boto3 requests-aws4auth

## Step 2: Add Permissions to the SageMaker Execution Role

Run the cell below to identify your execution role, then attach the `opensearch_bedrock_policy.json` policy to it in the IAM console.

In [ ]:
import boto3
import json

sts = boto3.client("sts")
caller = sts.get_caller_identity()
role_name = caller["Arn"].split("/")[1]

## Step 3: Create the OpenSearch Serverless Collection

This cell creates a network policy, a data access policy, and a vector search collection.

In [ ]:
import time

COLLECTION_NAME = "vector-demo"
REGION = "us-east-1"

aoss = boto3.client("opensearchserverless", region_name=REGION)

account_id = caller["Account"]
identity_arn = f"arn:aws:iam::{account_id}:role/{role_name}"

# Network policy - allows public access to the collection
try:
    aoss.create_security_policy(
        name=f"{COLLECTION_NAME}-net",
        type="network",
        policy=json.dumps([{
            "Rules": [
                {"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]},
                {"ResourceType": "dashboard", "Resource": [f"collection/{COLLECTION_NAME}"]}
            ],
            "AllowFromPublic": True
        }])
    )
    print("Network policy created.")
except aoss.exceptions.ConflictException:
    print("Network policy already exists.")

# Data access policy - grants index and document permissions to our role
try:
    aoss.create_access_policy(
        name=f"{COLLECTION_NAME}-access",
        type="data",
        policy=json.dumps([{
            "Rules": [
                {
                    "ResourceType": "index",
                    "Resource": [f"index/{COLLECTION_NAME}/*"],
                    "Permission": ["aoss:CreateIndex", "aoss:UpdateIndex", "aoss:DescribeIndex",
                                   "aoss:ReadDocument", "aoss:WriteDocument", "aoss:DeleteIndex"]
                },
                {
                    "ResourceType": "collection",
                    "Resource": [f"collection/{COLLECTION_NAME}"],
                    "Permission": ["aoss:CreateCollectionItems", "aoss:DescribeCollectionItems",
                                   "aoss:UpdateCollectionItems"]
                }
            ],
            "Principal": [identity_arn, f"arn:aws:sts::{account_id}:assumed-role/{role_name}/*"]
        }])
    )
    print("Data access policy created.")
except aoss.exceptions.ConflictException:
    print("Data access policy already exists.")

# Create the vector search collection
try:
    response = aoss.create_collection(
        name=COLLECTION_NAME,
        type="VECTORSEARCH",
        encryptionConfig={"aWSOwnedKey": True}
    )
    collection_id = response["createCollectionDetail"]["id"]
    print(f"Collection creation initiated. ID: {collection_id}")
except aoss.exceptions.ConflictException:
    collections = aoss.batch_get_collection(names=[COLLECTION_NAME])
    collection_id = collections["collectionDetails"][0]["id"]
    print(f"Collection already exists. ID: {collection_id}")

# Wait for the collection to become active
print("Waiting for collection to become active...", end="")
while True:
    status = aoss.batch_get_collection(ids=[collection_id])
    collection_status = status["collectionDetails"][0]["status"]
    if collection_status == "ACTIVE":
        collection_endpoint = status["collectionDetails"][0]["collectionEndpoint"]
        print(f" Done!")
        print(f"Endpoint: {collection_endpoint}")
        break
    print(".", end="", flush=True)
    time.sleep(10)

## Step 4: Connect to the Collection

Authenticate using IAM credentials and create an OpenSearch client.

In [ ]:
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth

credentials = boto3.Session().get_credentials()
aws_auth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    REGION,
    "aoss",
    session_token=credentials.token,
)

host = collection_endpoint.replace("https://", "")

client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=aws_auth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=60,
)

print("Connected to OpenSearch Serverless.")

## Step 5: Create Two Vector Indexes

We create separate indexes for two knowledge domains: product support documentation and HR policies. Each index uses cosine similarity with 1024-dimensional vectors to match the output of Amazon Titan Embed Text V2.

In [ ]:
support_index = "support-docs"
hr_index = "hr-policies"

index_body = {
    "settings": {
        "index.knn": True
    },
    "mappings": {
        "properties": {
            "content": {"type": "text"},
            "embedding": {
                "type": "knn_vector",
                "dimension": 1024,
                "space_type": "cosinesimil"
            }
        }
    }
}

response = client.indices.create(index=support_index, body=index_body)
print(f"Index '{support_index}' created: {response['acknowledged']}")

response = client.indices.create(index=hr_index, body=index_body)
print(f"Index '{hr_index}' created: {response['acknowledged']}")

## Step 6: Load and Chunk the Documents

Read each text file and split it into fixed-size chunks with overlap. This is a common production pattern for RAG systems. It ensures each chunk is small enough to embed meaningfully while the overlap preserves context at chunk boundaries.

In [ ]:
def fixed_size_chunk(text, chunk_size=500, overlap=50):
    """Split text into fixed-size word chunks with overlap."""
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

# Load the documents
with open("support_docs.txt", "r") as f:
    support_text = f.read()

with open("hr_policies.txt", "r") as f:
    hr_text = f.read()

# Chunk with 300 words per chunk and 20% overlap (60 words)
support_chunks = fixed_size_chunk(support_text, chunk_size=300, overlap=60)
hr_chunks = fixed_size_chunk(hr_text, chunk_size=300, overlap=60)

print(f"Support docs: {len(support_chunks)} chunks")
for i, chunk in enumerate(support_chunks):
    print(f"  Chunk {i+1}: {len(chunk.split())} words - \"{chunk[:80]}...\"")

print(f"\nHR policies: {len(hr_chunks)} chunks")
for i, chunk in enumerate(hr_chunks):
    print(f"  Chunk {i+1}: {len(chunk.split())} words - \"{chunk[:80]}...\"")

## Step 7: Generate Embeddings and Index the Documents

For each chunk, generate a vector embedding using Amazon Bedrock (Titan Embed Text V2), then store the chunk text along with its embedding in the appropriate index.

In [ ]:
bedrock = boto3.client("bedrock-runtime", region_name=REGION)


def get_embedding(text):
    """Generate a 1024-dimensional embedding using Titan Embed Text V2."""
    response = bedrock.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=json.dumps({
            "inputText": text,
            "dimensions": 1024,
            "normalize": True
        })
    )
    return json.loads(response["body"].read())["embedding"]

In [ ]:
def index_chunks(chunks, index_name):
    """Embed and index each chunk into the specified index."""
    for i, chunk in enumerate(chunks):
        embedding = get_embedding(chunk)

        document = {
            "content": chunk,
            "embedding": embedding
        }

        client.index(index=index_name, body=document)
        print(f"  Indexed chunk {i+1}/{len(chunks)}")

print(f"Indexing into '{support_index}':")
index_chunks(support_chunks, support_index)

print(f"\nIndexing into '{hr_index}':")
index_chunks(hr_chunks, hr_index)

print("\nDone!")

## Step 8: Search the Support Docs Index

Convert a natural language query into an embedding, then use k-NN search to find the most similar documents in the support index.

In [ ]:
import time
time.sleep(30)

query = "how do I reset my password"
query_vector = get_embedding(query)

search_body = {
    "size": 3,
    "_source": {"excludes": ["embedding"]},
    "query": {
        "knn": {
            "embedding": {
                "vector": query_vector,
                "k": 3
            }
        }
    }
}

print(f"Query: \"{query}\"")
print(f"Searching index: {support_index}\n")

response = client.search(index=support_index, body=search_body)

print("Results:")
for hit in response["hits"]["hits"]:
    print(f"  Score: {hit['_score']:.4f}")
    print(f"    {hit['_source']['content'][:150]}...")
    print()

## Step 9: Search the HR Policies Index

Run the same type of query against the HR policies index to find relevant policy information.

In [ ]:
query = "how much parental leave do I get"
query_vector = get_embedding(query)

search_body = {
    "size": 3,
    "_source": {"excludes": ["embedding"]},
    "query": {
        "knn": {
            "embedding": {
                "vector": query_vector,
                "k": 3
            }
        }
    }
}

print(f"Query: \"{query}\"")
print(f"Searching index: {hr_index}\n")

response = client.search(index=hr_index, body=search_body)

print("Results:")
for hit in response["hits"]["hits"]:
    print(f"  Score: {hit['_score']:.4f}")
    print(f"    {hit['_source']['content'][:150]}...")
    print()

## Step 10: RAG - Retrieve and Generate an Answer

Combine vector search with a foundation model to answer questions. First retrieve relevant chunks from the index, then pass them as context to Amazon Nova Lite via the Converse API.

In [ ]:
query = "can I work from home 4 days a week?"
query_vector = get_embedding(query)

# Retrieve the top 3 matching chunks from the HR index
search_body = {
    "size": 3,
    "_source": {"excludes": ["embedding"]},
    "query": {"knn": {"embedding": {"vector": query_vector, "k": 3}}}
}

results = client.search(index=hr_index, body=search_body)
context = "\n\n".join([hit["_source"]["content"] for hit in results["hits"]["hits"]])

# Generate an answer using Amazon Nova Lite via the Converse API
response = bedrock.converse(
    modelId="us.amazon.nova-lite-v1:0",
    messages=[
        {
            "role": "user",
            "content": [
                {"text": f"Answer the question based only on the following context:\n\n{context}\n\nQuestion: {query}"}
            ]
        }
    ],
    inferenceConfig={"maxTokens": 300, "temperature": 0.1}
)

answer = response["output"]["message"]["content"][0]["text"]
print(f"Query: \"{query}\"\n")
print(f"Answer: {answer}")

## Cleanup

Delete the indexes, collection, and policies created during this demo.

In [ ]:
# Delete the indexes
client.indices.delete(index=support_index)
print(f"Deleted index: {support_index}")
client.indices.delete(index=hr_index)
print(f"Deleted index: {hr_index}")

# Delete the collection
aoss.delete_collection(id=collection_id)
print(f"Deleted collection: {COLLECTION_NAME}")

# Delete policies
aoss.delete_security_policy(name=f"{COLLECTION_NAME}-net", type="network")
aoss.delete_access_policy(name=f"{COLLECTION_NAME}-access", type="data")
print("Deleted all policies.")